In [18]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
import dotenv
import os
from typing import TypedDict, List, Annotated, Optional, Literal
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field
import operator

In [19]:
dotenv.load_dotenv()

True

In [20]:
generator_llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("API_KEY"))
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("API_KEY"))
optimizer_llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("API_KEY"))

In [21]:
# schema for structured output

class TweetBaseModel(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Evaluation of the tweet")
    feedback: str = Field(..., description="Feedback about the tweet")


In [22]:
structured_eval_llm = evaluator_llm.with_structured_output(TweetBaseModel)

In [23]:
# state
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iterations: int
    tweet_history: Annotated[List[str], operator.add]
    feedback_history: Annotated[List[str], operator.add]


In [24]:
# create functions for each node
def generate_tweet(state: TweetState) -> TweetState:
    messages = [
    SystemMessage(content="You are a funny and clever Twitter/X influencer."),
    HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
This is version {state['iteration'] + 1}.
""")
]

    # send generator llm
    response = generator_llm.invoke(messages).content

    # return response
    return {'tweet': response, 'tweet_history': [response]}

In [25]:
def evaluate_tweet(state: TweetState) -> TweetState:
    message = [
        SystemMessage(
            content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
        ),
        HumanMessage(
            content=f"""
Evaluate the following tweet:
tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

- Originality - Is this fresh, or have you seen it a hundred times before?
- Humor - Did it genuinely make you smile, laugh, or chuckle?
- Punchiness - Is it short, sharp, and scroll-stopping?
- Virality Potential - Would people retweet or share it?
- Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "what happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Don’t end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces of the auntie-uncle universe" or vague summaries)

## Respond ONLY in structured format:
evaluation: "approved" or "needs_improvement"
feedback: One paragraph explaining the strengths and weaknesses
""")
]
    structured_response = structured_eval_llm.invoke(message)
    return {'evaluation': structured_response.evaluation, 'feedback': structured_response.feedback, 'feedback_history': [structured_response.feedback]}

In [26]:
def optimize_tweet(state: TweetState) -> TweetState:
    messages = [
    SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
    HumanMessage(content=f"""
        Improve the tweet based on this feedback:
        "{state['feedback']}"

        Topic: "{state['topic']}"
        Original Tweet:
        {state['tweet']}

        Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
        """)
        ]
    iteration = state['iteration'] + 1
    response = optimizer_llm.invoke(messages).content
    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [27]:
# conditon based routing based on evaluation model response

def route_evaluation(state: TweetState) -> str:
    if state['evaluation'] == 'approved':
        return 'approved'
    elif state['evaluation'] == 'needs_improvement' and state['iteration'] < state['max_iterations']:
        return 'needs_improvement'
    else:
        return 'stop'

In [28]:
graph = StateGraph(TweetState)

# add nodes

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)


# add egdges

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges(
    'evaluate',
    route_evaluation,
    {'approved': END, 'needs_improvement': 'optimize', 'stop': END},
)

# cycle loop only manipulating edges
graph.add_edge('optimize', 'evaluate')

wf = graph.compile()

In [39]:
# testing

initial_state = {

    'topic': 'IIT',
    'iteration': 1,
    'max_iterations': 5
}

wf.invoke(initial_state)


c:\Users\acer\Udit\langgraph_proj\env\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=TweetBaseModel(evaluation...e engaging the reader."), input_type=TweetBaseModel])
  return self.__pydantic_serializer__.to_python(


{'topic': 'IIT',
 'tweet': 'IIT: where you can learn quantum mechanics and still not understand why your roommate thinks 3 AM is the perfect time for a snack debate. I mean, who needs sleep when you can argue about the existential crisis of instant noodles? 🍜💔 #StudentLife #IITLogic',
 'evaluation': 'approved',
 'feedback': "This tweet showcases originality by blending the unique experiences of IIT students with relatable themes of late-night debates and existential musings about instant noodles. The humor is effective in its absurdity, making it amusing and relatable without feeling forced. Its punchiness is commendable, as it conveys a vivid scenario without unnecessary fluff, capturing attention quickly. The potential for virality is high due to its clever commentary on student life, appealing to a broad audience, especially within academia. The format adheres to Twitter's structure, maintaining clarity and brevity while engaging the reader.",
 'iteration': 1,
 'max_iterations': 5,
